In [3]:
from pathlib import Path
import numpy as np
import pandas as pd

import os

import json

from nilearn.maskers import NiftiLabelsMasker
from neuroginius.atlas import Atlas #not strictly necessary, can use path to a parcellation atlas instead

movie_name = "AfterTheRain"


In [4]:
atlas = Atlas().from_name("schaefer200")
masker = NiftiLabelsMasker(labels_img=atlas.maps, standardize="zscore_sample")
masker.fit()

NiftiLabelsMasker(labels_img='/homes_unix/agillig/nilearn_data/schaefer_2018/Schaefer2018_200Parcels_7Networks_order_FSLMNI152_2mm.nii.gz',
                  standardize='zscore_sample')

In [41]:
data_path.parent.parent

PosixPath('../../data/emofilm_fmri')

In [50]:
data_path = Path("../../data/emofilm_fmri/derivatives/preprocessing/")
out_dir = Path(f"../../data/emofilm_fmri/derivatives/parcellated_timeseries/{atlas.name}/{movie_name}")
os.makedirs(out_dir, exist_ok=True)

subjects_list = []
for sub_idx in range(25):
    sub_str = f"S{sub_idx+1:02d}"

    files = list(
        (data_path / f"sub-{sub_str}").glob(
            f"ses-*/func/sub-{sub_str}_ses-*_task-{movie_name}_space-MNI_desc-ppres_bold.nii.gz"
        )
    )

    if not files:
        continue

    # file = files[0]  # or handle multiple matches

   
    # subjects_list.append(sub_str)
    # print(f"Processing {file}...")
    # parcellated_time_series = masker.transform(file)
    
    # save_path = out_dir / f"sub-{sub_str}_ses-3_task-{movie_name}_space-MNI_desc-ppres_bold_parcellated.tsv"

    # np.savetxt(save_path, parcellated_time_series)

    event_timings_files = list(
        (Path("../../data/emofilm_fmri") / f"sub-{sub_str}").glob(
            f"ses-*/func/sub-{sub_str}_ses-*_task-scan_acq-{movie_name}_events.tsv"
        )
    )
    working_dir = Path("../../data/emofilm/event_timings")
    os.makedirs(working_dir, exist_ok=True)
    event_timings = pd.read_csv(event_timings_files[0], sep="\t") if event_timings_files else None
    event_timings.to_csv(working_dir / f"sub-{sub_str}_{movie_name}_events.tsv", index=False)


In [51]:
event_timings

,onset,duration,trial_type
0,2.788953,89.978769,rest
1,93.912097,496.081650,film
2,591.043606,89.965891,rest


In [22]:
behav_data_path = Path("../../data/emofilm_annotation/derivatives/")
os.listdir(behav_data_path)


with open(behav_data_path / f"Annot_{movie_name}_stim.json", "r") as f:
    metadata = json.load(f)

behav_data = pd.read_csv(behav_data_path / f"Annot_{movie_name}_stim.tsv.gz",
    compression="gzip",
    sep="\t",
    header=None
    )

behav_data.columns = metadata["Columns"]


working_dir = Path("../../data/emofilm/")
behav_data.to_csv(working_dir / f"Annot_{movie_name}_stim.csv", index=False)


In [23]:
behav_data.head()

,Standards,PleasantSelf,SocialNorms,PleasantOther,GoalsOther,Controlled,Predictable,Suddenly,Agent,Urgency,...,Disgust,Happiness,Fear,Regard,Anxiety,Satisfaction,Pride,Surprise,Love,Sad
0,0.105069,0.453663,0.237408,0.424665,-0.388207,0.097503,1.862368,-0.079510,-0.418996,-0.208150,...,-0.043765,0.471479,-0.308988,-0.133703,-0.855552,0.246556,0.328818,-0.295054,0.333593,-0.470214
1,0.024357,0.449826,0.238010,0.424665,-0.388207,0.097503,1.747290,-0.134586,-0.418996,-0.737396,...,-0.031966,0.471479,-0.308988,-0.133703,-0.856457,0.246556,0.328818,-0.295054,0.333593,-0.470214
2,0.070704,0.414445,0.293824,0.424665,-0.291213,0.097503,1.504896,-0.188682,-0.418996,-0.999275,...,-0.031966,0.471479,-0.588825,-0.133703,-1.210829,0.246556,0.328818,-0.295054,0.333593,-0.470214
3,0.105296,0.413923,0.292980,0.406338,-0.101414,0.097503,1.435084,-0.296824,-0.418996,-1.084361,...,-0.031966,0.432859,-0.873184,-0.133703,-1.495191,0.246556,0.328818,-0.295054,0.179468,-0.470214
4,-0.019593,0.413923,0.371977,0.218044,0.058179,0.097503,1.354356,-0.425361,-0.418996,-1.121516,...,-0.402893,0.326432,-0.873184,-0.133703,-1.831240,0.217631,0.328818,-0.295054,-0.612967,-0.469630


create appropriate data structure for training: list of np arrays of shape (n_timepoints x n_regions)

In [28]:
X = []
subjects_list = []
for sub_idx in range(25):
    sub_str = f"S{sub_idx+1:02d}"

    file = out_dir / f"sub-{sub_str}_ses-3_task-{movie_name}_space-MNI_desc-ppres_bold_parcellated.tsv"


    if not os.path.exists(file):
        continue

    tmp_array = np.loadtxt(file)
    X.append(tmp_array)
    


In [31]:
np.savez(working_dir/f"{movie_name}_parcellated_timeseries.npz",*X)